# Advanced: Multi-Strategy Record Suppression with PAMOLA.CORE

**Goal**: Master all 5 record suppression conditions using operation.execute()

**What you'll learn:**
- **Strategy 1**: Value-based record removal (specific diagnoses)
- **Strategy 2**: Risk-based removal (k-anonymity threshold)
- **Strategy 3**: Custom multi-condition removal (complex AND/OR logic)
- Calculate advanced privacy metrics (k-anonymity, l-diversity)
- Analyze record retention rates
- Production deployment patterns with audit trails

**Prerequisites:**
- Completed the simple notebook
- Understanding of privacy concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/suppression/
        └── 02_record_suppression_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.anonymization.suppression.record_op import RecordSuppressionOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record patient dataset
- Auto-creates sample if file not found
- Review data structure and identify sensitive records

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 8 columns)
- Sample records (first 5 rows)
- Column statistics:
  - Categorical fields: diagnosis (7 types), zipcode, risk_group
  - Numeric fields: age (18-85), risk_score (1-10), treatment_cost
  - Rare diagnoses for demonstration (<5% frequency)

**Note:** Generated data includes realistic patterns for all suppression strategies

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Diagnoses with rare diseases (for value-based suppression)
    common_diagnoses = ['Diabetes', 'Hypertension', 'Asthma', 'None', 'Flu']
    rare_diagnoses = ['Rare_Disease_X', 'Rare_Disease_Y']
    all_diagnoses = (
        list(np.random.choice(common_diagnoses, n_records - 30)) +
        list(np.random.choice(rare_diagnoses, 30))
    )
    np.random.shuffle(all_diagnoses)
    
    # Risk scores (1-10, for risk-based suppression)
    # Lower scores = higher risk (vulnerable to re-identification)
    risk_scores = np.concatenate([
        np.random.randint(1, 4, 50),   # High risk (< 4)
        np.random.randint(4, 7, 300),  # Medium risk
        np.random.randint(7, 11, 650)  # Low risk
    ])
    np.random.shuffle(risk_scores)
    
    # Ages
    ages = np.random.randint(18, 86, n_records)
    
    # Zipcodes (for custom conditions)
    zipcodes = np.random.choice(
        ['10001', '10002', '10003', '10004', '10005', '10006', '10007'],
        n_records
    )
    
    # Risk groups
    risk_groups = np.random.choice(['Low', 'Medium', 'High', 'Critical'], n_records)
    
    # Treatment costs
    treatment_costs = np.random.randint(500, 10000, n_records)
    
    df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'patient_id': [f'P{i:05d}' for i in range(1, n_records + 1)],
        'age': ages,
        'diagnosis': all_diagnoses,
        'risk_score': risk_scores,
        'zipcode': zipcodes,
        'risk_group': risk_groups,
        'treatment_cost': treatment_costs
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

print(f"\n💡 Perfect dataset for testing all 3 strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field names for each strategy
   - `strategy1_field`: For value-based suppression
   - `strategy2_field`: For risk-based suppression
   - `strategy3_field`: For custom multi-condition
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field)
- Value distributions per field (top 5 values)
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource created (✓)

**Note:** All fields must exist in dataset before executing strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_field": "remote_preference",      # Value-based suppression
    "strategy2_field": "years_of_experience",     # Risk-based suppression
    "strategy3_field": "location_city"             # Custom multi-condition
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    
    print(f"  ✓ {strategy:20s}: '{field_name}'")
    print(f"      Unique: {df[field_name].nunique()}")
    print(f"      Distribution (top 5):")
    top5 = df[field_name].value_counts().head(5)
    for val, count in top5.items():
        print(f"         {str(val):20s}: {count:4d} ({count/len(df)*100:.1f}%)")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_record_001",
    task_type="multi_strategy",
    description="Multi-strategy record suppression",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Value-Based Record Suppression

**How to use:**
- Removes entire records with specific values
- Targets rare diseases for privacy protection
- Run to execute strategy

**Key parameters:**
- `suppression_condition='value'`: Match specific values
- `suppression_values=['suppression_values']`: Values to suppress
- `save_suppressed_records=True`: Audit trail of removed records
- `suppression_mode='REMOVE'`: Delete entire rows

**What you'll see:**
- Configuration confirmation
- Progress bar: validation → check → remove → metrics → viz → save
- Completion message with execution time (1-3 seconds)
- Record counts (original, remaining, suppressed)

**Note:** Creates separate file for suppressed records with suppression reasons

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: VALUE-BASED RECORD SUPPRESSION")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Value-based",
    unit="steps",
    track_memory=True,
    level=0
)

# Define values to suppress
values_suppress = ['Remote']

# Create operation
operation_s1 = RecordSuppressionOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy1_field"],
    suppression_mode='REMOVE',
    
    # Value-based suppression
    suppression_condition='value',
    suppression_values=values_suppress,
    
    # Audit trail
    save_suppressed_records=True,
    suppression_reason_field='_suppression_reason',
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  Suppressing values: {values_suppress}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_value_based',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    [f for f in (task_dir / 'strategy1_value_based' / 'output').glob('*.csv') if 'suppressed' not in f.name],
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    field_s1 = FIELD_CONFIG["strategy1_field"]
    
    original_count = len(df)
    remaining_count = len(result_df_s1)
    suppressed_count = original_count - remaining_count
    
    print(f"\n📊 Record Suppression:")
    print(f"   Original:   {original_count:,} records")
    print(f"   Remaining:  {remaining_count:,} records")
    print(f"   Suppressed: {suppressed_count:,} records ({(suppressed_count/original_count*100):.1f}%)")
    
    # Check suppressed records file
    suppressed_dir_s1 = task_dir / 'strategy1_value_based' / 'output' / 'suppressed_records'
    if suppressed_dir_s1.exists():
        suppressed_files = list(suppressed_dir_s1.glob('*.csv'))
        if suppressed_files:
            print(f"\n📁 Suppressed records saved: {len(suppressed_files)} file(s)")
            print(f"   Location: {suppressed_dir_s1}")

## STRATEGY 2: Risk-Based Record Suppression

**How to use:**
- Removes records with k-anonymity risk below threshold
- Protects vulnerable records from re-identification
- Run to execute strategy

**Key parameters:**
- `suppression_condition='risk'`: Use risk threshold
- `ka_risk_field=ka_risk_field`: Field containing risk scores
- `risk_threshold=4`: Remove records with score < 4
- `save_suppressed_records=True`: Keep audit trail

**What you'll see:**
- Configuration confirmation
- Progress bar: validation → risk check → remove → metrics → viz → save
- Completion message with execution time (1-3 seconds)
- High-risk record counts and suppression rate

**Note:** Lower risk scores = higher re-identification risk = removal priority

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: RISK-BASED RECORD SUPPRESSION")
print("=" * 80 + "\n")

ka_risk_field = 'specialization'

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Risk-based",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = RecordSuppressionOperation(
    field_name=FIELD_CONFIG["strategy2_field"],
    suppression_mode='REMOVE',
    
    # Risk-based suppression
    suppression_condition='risk',
    ka_risk_field=ka_risk_field,
    risk_threshold=4,  # Remove records with risk_score < 4
    
    # Audit trail
    save_suppressed_records=True,
    suppression_reason_field='_suppression_reason',
    
    output_format='csv',
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"  Risk threshold: < {operation_s2.risk_threshold}")
print(f"  Risk field: '{operation_s2.ka_risk_field}'")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_risk_based',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results
output_files_s2 = sorted(
    [f for f in (task_dir / 'strategy2_risk_based' / 'output').glob('*.csv') if 'suppressed' not in f.name],
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    
    # Note: result_df_s2 is processed from original_count
    original_count = len(df)
    remaining_count = len(result_df_s2)
    suppressed_count = original_count - remaining_count
    
    print(f"\n📊 Record Suppression (after Strategy 1):")
    print(f"   Input:      {original_count:,} records")
    print(f"   Remaining:  {remaining_count:,} records")
    print(f"   Suppressed: {suppressed_count:,} records ({(suppressed_count/original_count*100):.1f}%)")
    
    # Show risk distribution
    print(f"\n📈 Risk Score Distribution (remaining records):")
    if ka_risk_field in result_df_s2.columns:
        risk_dist = result_df_s2[ka_risk_field].value_counts().sort_index()
        for score, count in risk_dist.items():
            print(f"   Score {score}: {count:4d} records")

## STRATEGY 3: Custom Multi-Condition Suppression

**How to use:**
- Removes records matching complex AND/OR conditions
- Combines multiple field criteria
- Run to execute strategy

**Key parameters:**
- `suppression_condition='custom'`: Use multi-field logic
- `multi_conditions`: List of condition dictionaries
- `condition_logic='AND'`: Both conditions must match

**What you'll see:**
- Configuration confirmation with conditions
- Progress bar: validation → evaluate → remove → metrics → viz → save
- Completion message with execution time (1-3 seconds)
- Complex condition matching statistics

**Note:** Powerful for compliance requirements (GDPR right to be forgotten, etc.)

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: CUSTOM MULTI-CONDITION SUPPRESSION")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Custom",
    unit="steps",
    track_memory=True,
    level=0
)

# Define complex conditions
multi_conditions = [
    {
        'field': 'years_of_experience',
        'operator': 'gt',
        'values': [10]
    },
    {
        'field': 'location_province',
        'operator': 'in',
        'values': ['BC', 'NT']
    }
]

operation_s3 = RecordSuppressionOperation(
    field_name=FIELD_CONFIG["strategy3_field"],
    suppression_mode='REMOVE',
    
    # Custom multi-condition suppression
    suppression_condition='custom',
    multi_conditions=multi_conditions,
    condition_logic='AND',  # Both conditions must be true
    
    # Audit trail
    save_suppressed_records=True,
    suppression_reason_field='_suppression_reason',
    
    output_format='csv',
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  Conditions ({operation_s3.condition_logic}):")
for i, cond in enumerate(multi_conditions, 1):
    print(f"    {i}. {cond['field']} {cond['operator']} {cond['values']}")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_custom',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results
output_files_s3 = sorted(
    [f for f in (task_dir / 'strategy3_custom' / 'output').glob('*.csv') if 'suppressed' not in f.name],
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    
    # Note: result_df_s3 is processed from original_count
    original_count = len(df)
    remaining_count = len(result_df_s3)
    suppressed_count = original_count - remaining_count
    
    print(f"\n📊 Record Suppression (after Strategy 2):")
    print(f"   Input:      {original_count:,} records")
    print(f"   Remaining:  {remaining_count:,} records")
    print(f"   Suppressed: {suppressed_count:,} records ({(suppressed_count/original_count*100):.1f}%)")

    # Verify conditions
    print(f"\n✓ Verification: All remaining records satisfy:")

    condition_checks = []

    for i, cond in enumerate(multi_conditions, 1):
        field = cond.get('field')
        operator = cond.get('operator')
        value = cond.get('values')

        # Missing column → safe default (condition False)
        if field not in result_df_s3.columns:
            mask = pd.Series(False, index=result_df_s3.index)
            desc = f"{field} {operator} {value} (missing column)"
            count = 0
            print(f"   Condition {i} ({desc}): {count} records (skipped)")
            condition_checks.append(mask)
            continue

        series = result_df_s3[field]

        # Numeric operators
        if operator in {'gt', 'lt', 'gte', 'lte'}:
            s_num = pd.to_numeric(series, errors='coerce')

            if isinstance(value, (list, tuple)):
                # Use first element if list provided (defensive)
                value_cmp = value[0] if value else None
            else:
                value_cmp = value

            if value_cmp is None:
                mask = pd.Series(False, index=result_df_s3.index)
            else:
                mask = {
                    'gt': s_num > value_cmp,
                    'lt': s_num < value_cmp,
                    'gte': s_num >= value_cmp,
                    'lte': s_num <= value_cmp,
                }[operator]

            desc = f"{field} {operator} {value_cmp}"

        elif operator == 'eq':
            mask = series == value
            desc = f"{field} == {value}"

        elif operator == 'ne':
            mask = series != value
            desc = f"{field} != {value}"

        elif operator == 'in':
            values = value if isinstance(value, (list, tuple, set)) else [value]
            mask = series.astype(str).isin([str(v) for v in values])
            desc = f"{field} in {values}"

        elif operator == 'not_in':
            values = value if isinstance(value, (list, tuple, set)) else [value]
            mask = ~series.astype(str).isin([str(v) for v in values])
            desc = f"{field} not in {values}"

        else:
            mask = pd.Series(False, index=result_df_s3.index)
            desc = f"{field} {operator} {value} (unknown operator)"

        condition_checks.append(mask)
        count = int(mask.sum())
        print(
            f"   Condition {i} ({desc}): "
            f"{count} records (should be 0 if {operation_s3.condition_logic} logic worked)"
        )

    # Check if all conditions are satisfied together
    if operation_s3.condition_logic == 'AND':
        all_conditions = pd.Series([True] * len(result_df_s3))
        for mask in condition_checks:
            all_conditions &= mask
        both_count = all_conditions.sum()
        print(f"   All {len(multi_conditions)} conditions together (AND): {both_count} records (should be 0)")
    elif operation_s3.condition_logic == 'OR':
        any_condition = pd.Series([False] * len(result_df_s3))
        for mask in condition_checks:
            any_condition |= mask
        any_count = any_condition.sum()
        print(f"   Any condition (OR): {any_count} records (should be 0)")  

## Step 4: Strategy Comparison

**How to use:**
- Review execution times for all 3 strategies
- Compare suppression rates and effectiveness
- Identify best strategy for your use case

**What you'll see:**
- Execution time for each strategy (seconds)
- Cumulative suppression (progressive record reduction)
- Total processing time for all strategies
- Final retention rate (% records remaining)

**Strategy selection guide:**
- **Value-Based**: Remove known sensitive/rare categories
- **Risk-Based**: Automated privacy protection via k-anonymity
- **Custom**: Compliance-driven removal (GDPR, HIPAA, etc.)

**Note:** Strategies are cumulative - each processes output from previous strategy

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Value-based):  {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Risk-based):   {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Custom):       {elapsed_s3:6.2f}s")
print(f"  Total:                     {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print(f"\n📈 Cumulative Suppression (Progressive):")
original_count = len(df)
if 'result_df_s1' in locals():
    s1_remaining = len(result_df_s1)
    s1_suppressed = original_count - s1_remaining
    print(f"  After Strategy 1: {s1_remaining:,} records ({(s1_remaining/original_count*100):.1f}% retained)")
    print(f"    Suppressed: {s1_suppressed:,} records ({(s1_suppressed/original_count*100):.1f}%)")

if 'result_df_s2' in locals():
    s2_remaining = len(result_df_s2)
    s2_suppressed = s1_remaining - s2_remaining
    print(f"\n  After Strategy 2: {s2_remaining:,} records ({(s2_remaining/original_count*100):.1f}% retained)")
    print(f"    Suppressed: {s2_suppressed:,} records ({(s2_suppressed/s1_remaining*100):.1f}% of input)")

if 'result_df_s3' in locals():
    s3_remaining = len(result_df_s3)
    s3_suppressed = s2_remaining - s3_remaining
    total_suppressed = original_count - s3_remaining
    print(f"\n  After Strategy 3: {s3_remaining:,} records ({(s3_remaining/original_count*100):.1f}% retained)")
    print(f"    Suppressed: {s3_suppressed:,} records ({(s3_suppressed/s2_remaining*100):.1f}% of input)")
    
    print(f"\n✨ FINAL TOTALS:")
    print(f"  Original:        {original_count:,} records")
    print(f"  Final:           {s3_remaining:,} records")
    print(f"  Total Suppressed: {total_suppressed:,} records ({(total_suppressed/original_count*100):.1f}%)")
    print(f"  Retention Rate:   {(s3_remaining/original_count*100):.1f}%")

## Step 5: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Auto-loads NEWEST files from each category
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: Suppression condition, records suppressed, rate
2. **Output Data**: Preview with remaining records
3. **Suppressed Records**: Audit file with removal reasons
4. **Visualizations**: Before/after comparison charts (latest batch only)

**Validate:**
- ✅ Target records properly removed
- ✅ Suppressed records saved with reasons
- ✅ Remaining data matches criteria
- ✅ Audit trail complete for compliance

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_value_based', 'Strategy 1: Value-Based Suppression'),
    ('strategy2_risk_based', 'Strategy 2: Risk-Based Suppression'),
    ('strategy3_custom', 'Strategy 3: Custom Multi-Condition')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # Display key metrics
                print(f"   Operation:           {metrics.get('operation_type', 'N/A')}")
                print(f"   Condition:           {metrics.get('suppression_condition', 'N/A')}")
                print(f"   Records Suppressed:  {metrics.get('records_suppressed', 0):,}")
                print(f"   Suppression Rate:    {metrics.get('suppression_rate', 0):.2f}%")
                print(f"   Remaining Records:   {metrics.get('remaining_records', 0):,}")
                print(f"   Duration:            {metrics.get('duration_seconds', 0):.4f}s")
                
                # Strategy-specific metrics
                if 'suppression_values_count' in metrics:
                    print(f"   Values Suppressed:   {metrics.get('suppression_values_count', 0)}")
                if 'risk_threshold' in metrics:
                    print(f"   Risk Threshold:      {metrics.get('risk_threshold', 0)}")
                if 'multi_conditions_count' in metrics:
                    print(f"   Conditions:          {metrics.get('multi_conditions_count', 0)}")
                    print(f"   Logic:               {metrics.get('condition_logic', 'N/A')}")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Output Data Preview (NEWEST)
    output_dir = strategy_dir / 'output'
    if output_dir.exists():
        output_files = sorted(
            [f for f in output_dir.glob('*.csv') if 'suppressed' not in f.name],
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        if output_files:
            print(f"\n📄 OUTPUT: {output_files[0].name}")
            try:
                output_df = pd.read_csv(output_files[0])
                print(f"   Records: {len(output_df):,}")
                print(f"\n   Preview (first 5 rows):")

                preview_cols = [FIELD_CONFIG["strategy1_field"], FIELD_CONFIG["strategy2_field"], FIELD_CONFIG["strategy3_field"]]
                existing_cols = [c for c in preview_cols if c in output_df.columns]
                missing_cols = [c for c in preview_cols if c not in output_df.columns]

                if not existing_cols:
                    print("   ⚠️  None of the preview columns exist in output")
                    print(output_df.head().to_string(index=False))
                else:
                    print(output_df[existing_cols].head().to_string(index=False))

            except Exception as e:
                print(f"   ⚠️  Error loading output: {e}")
    
    # 3. Suppressed Records (NEWEST)
    suppressed_dir = strategy_dir / 'output' / 'suppressed_records'
    if suppressed_dir.exists():
        suppressed_files = sorted(
            list(suppressed_dir.glob('*.csv')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        if suppressed_files:
            print(f"\n📄 SUPPRESSED RECORDS: {len(suppressed_files)} file(s)")
            try:
                supp_df = pd.read_csv(suppressed_files[0])
                print(f"   Total suppressed: {len(supp_df):,} records")
                if '_suppression_reason' in supp_df.columns:
                    reason_counts = supp_df['_suppression_reason'].value_counts()
                    print(f"\n   Suppression Reasons:")
                    for reason, count in reason_counts.items():
                        print(f"      {count:3d} records: {reason}")
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 4. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Calculate Privacy Metrics

**How to use:**
- Calculate k-anonymity for original and processed data
- Compare privacy improvements across strategies
- Run AFTER all 3 strategies complete

**What you'll see:**
- Original data: min k, avg k, vulnerable group count
- After each strategy: progressive k-anonymity improvements
- Final k-anonymity level (e.g., "7-anonymous")

**Privacy targets:**
- Min k ≥ 5: Basic privacy protection
- Min k ≥ 10: Strong privacy protection
- Zero vulnerable groups: Ideal state

**Note:** Record suppression directly improves k-anonymity by removing unique/rare records

In [ ]:
print("\n" + "=" * 80)
print("🔒 PRIVACY METRICS")
print("=" * 80 + "\n")

def calculate_k_anonymity(df, quasi_identifiers):
    """Calculate k-anonymity metrics"""
    # Filter to only existing columns
    existing_qis = [q for q in quasi_identifiers if q in df.columns]
    if not existing_qis:
        return None

    group_sizes = df.groupby(existing_qis).size()
    return {
        'min_k': int(group_sizes.min()),
        'avg_k': float(group_sizes.mean()),
        'vulnerable_groups': int((group_sizes < 5).sum())
    }

# Use diagnosis and zipcode as quasi-identifiers
quasi_ids = ['location_city']

try:
    # Original data
    k_orig = calculate_k_anonymity(df, quasi_ids)
    print(f"📊 ORIGINAL DATA:")
    print(f"   Quasi-identifiers: {', '.join(quasi_ids)}")
    print(f"   Min k: {k_orig['min_k']}")
    print(f"   Avg k: {k_orig['avg_k']:.2f}")
    print(f"   Vulnerable groups: {k_orig['vulnerable_groups']}")
    
    # After Strategy 1
    if 'result_df_s1' in locals() and result_df_s1 is not None:
        k_s1 = calculate_k_anonymity(result_df_s1, quasi_ids)
        print(f"\n✨ AFTER STRATEGY 1 (Value-Based):")
        print(f"   Min k: {k_s1['min_k']} ({k_s1['min_k']/k_orig['min_k']:.1f}x)")
        print(f"   Avg k: {k_s1['avg_k']:.2f} ({k_s1['avg_k']/k_orig['avg_k']:.1f}x)")
        print(f"   Vulnerable groups: {k_s1['vulnerable_groups']}")
    
    # After Strategy 2
    if 'result_df_s2' in locals() and result_df_s2 is not None:
        k_s2 = calculate_k_anonymity(result_df_s2, quasi_ids)
        print(f"\n✨ AFTER STRATEGY 2 (Risk-Based):")
        print(f"   Min k: {k_s2['min_k']} ({k_s2['min_k']/k_orig['min_k']:.1f}x)")
        print(f"   Avg k: {k_s2['avg_k']:.2f} ({k_s2['avg_k']/k_orig['avg_k']:.1f}x)")
        print(f"   Vulnerable groups: {k_s2['vulnerable_groups']}")
    
    # After Strategy 3
    if 'result_df_s3' in locals() and result_df_s3 is not None:
        k_s3 = calculate_k_anonymity(result_df_s3, quasi_ids)
        print(f"\n✨ AFTER STRATEGY 3 (Custom):")
        print(f"   Min k: {k_s3['min_k']} ({k_s3['min_k']/k_orig['min_k']:.1f}x)")
        print(f"   Avg k: {k_s3['avg_k']:.2f} ({k_s3['avg_k']/k_orig['avg_k']:.1f}x)")
        print(f"   Vulnerable groups: {k_s3['vulnerable_groups']}")
        
        print(f"\n🎯 Final dataset is {k_s3['min_k']}-anonymous")
        
    if not any(var in locals() for var in ['result_df_s1', 'result_df_s2', 'result_df_s3']):
        print("\n⚠️  No strategy results available - run all strategies first!")
        
except Exception as e:
    print(f"⚠️  Error calculating privacy metrics: {e}")
    print("   Make sure all strategies have been executed.")

## Step 7: Export Final Data

**How to use:**
- Export final processed datasets from all strategies
- Each strategy gets its own output folder
- Run AFTER all 3 strategies complete

**What you'll see (per strategy):**
- Export confirmation (file path, row count)
- Preview of first 5 rows
- Suppression statistics (records removed, rate)
- Suppressed records file location

**Output structure:**
```
advanced_output/
├── strategy1_value_based/
│   ├── remaining_records.csv
│   └── suppressed_records/
├── strategy2_risk_based/
│   ├── remaining_records.csv
│   └── suppressed_records/
└── strategy3_custom/
    ├── remaining_records.csv
    └── suppressed_records/
```

**Note:** Suppressed records saved separately for audit and compliance

In [ ]:
import os
from pathlib import Path

print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

print(f"\n📂 Export directory: {task_dir}\n")

# Strategy 1: Value-Based
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: VALUE-BASED SUPPRESSION")
    print("=" * 80)
    
    s1_dir = task_dir / 'strategy1_value_based'
    
    # Already saved by operation, just report
    output_files = list((s1_dir / 'output').glob('*.csv'))
    output_files = [f for f in output_files if 'suppressed' not in f.name]
    
    if output_files:
        print(f"\n✅ Output: {output_files[0].name}")
        print(f"   Location: {output_files[0]}")
        print(f"   Records: {len(result_df_s1):,}")
        
        print(f"\n📊 Preview (first 5 rows):")
        print(result_df_s1.head())
        
        suppressed = len(df) - len(result_df_s1)
        print(f"\n📈 Suppression Statistics:")
        print(f"   Original:   {len(df):,} records")
        print(f"   Remaining:  {len(result_df_s1):,} records")
        print(f"   Suppressed: {suppressed:,} records ({(suppressed/len(df)*100):.2f}%)")
        
        # Check for suppressed records
        supp_dir = s1_dir / 'output' / 'suppressed_records'
        if supp_dir.exists():
            supp_files = list(supp_dir.glob('*.csv'))
            if supp_files:
                print(f"\n📁 Suppressed records: {len(supp_files)} file(s)")
                print(f"   Location: {supp_dir}")
else:
    print("\n⚠️  Strategy 1 data not available")

# Strategy 2: Risk-Based
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: RISK-BASED SUPPRESSION")
    print("=" * 80)
    
    s2_dir = task_dir / 'strategy2_risk_based'
    
    output_files = list((s2_dir / 'output').glob('*.csv'))
    output_files = [f for f in output_files if 'suppressed' not in f.name]
    
    if output_files:
        print(f"\n✅ Output: {output_files[0].name}")
        print(f"   Location: {output_files[0]}")
        print(f"   Records: {len(result_df_s2):,}")
        
        print(f"\n📊 Preview (first 5 rows):")
        print(result_df_s2.head())
        
        s1_count = len(result_df_s1)
        suppressed = s1_count - len(result_df_s2)
        print(f"\n📈 Suppression Statistics (from Strategy 1 output):")
        print(f"   Input:      {s1_count:,} records")
        print(f"   Remaining:  {len(result_df_s2):,} records")
        print(f"   Suppressed: {suppressed:,} records ({(suppressed/s1_count*100):.2f}%)")
else:
    print("\n⚠️  Strategy 2 data not available")

# Strategy 3: Custom
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: CUSTOM MULTI-CONDITION")
    print("=" * 80)
    
    s3_dir = task_dir / 'strategy3_custom'
    
    output_files = list((s3_dir / 'output').glob('*.csv'))
    output_files = [f for f in output_files if 'suppressed' not in f.name]
    
    if output_files:
        print(f"\n✅ Output: {output_files[0].name}")
        print(f"   Location: {output_files[0]}")
        print(f"   Records: {len(result_df_s3):,}")
        
        print(f"\n📊 Preview (first 5 rows):")
        print(result_df_s3.head())
        
        s2_count = len(result_df_s2)
        suppressed = s2_count - len(result_df_s3)
        print(f"\n📈 Suppression Statistics (from Strategy 2 output):")
        print(f"   Input:      {s2_count:,} records")
        print(f"   Remaining:  {len(result_df_s3):,} records")
        print(f"   Suppressed: {suppressed:,} records ({(suppressed/s2_count*100):.2f}%)")
else:
    print("\n⚠️  Strategy 3 data not available")

# Summary
print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 All files saved to: {task_dir}")

if all(var in locals() for var in ['elapsed_s1', 'elapsed_s2', 'elapsed_s3']):
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total time: {total_time:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 3 suppression strategies implemented and compared
- ✅ Value-based removal of sensitive categories
- ✅ Risk-based removal via k-anonymity threshold
- ✅ Custom multi-condition removal with AND/OR logic
- ✅ Privacy metrics calculated (k-anonymity improvements)
- ✅ Complete audit trails with suppression reasons
- ✅ Production-ready artifacts generated

**Key takeaways:**
- **Record suppression** removes entire rows for maximum privacy
- **Value-based** targets known sensitive/rare categories
- **Risk-based** uses k-anonymity scores for automated protection
- **Custom conditions** enable compliance-driven removal (GDPR, HIPAA)
- Strategies are cumulative - each processes previous output
- Suppressed records saved separately for audit/legal requirements

**Strategy recommendations:**
- **Use Strategy 1 (Value-Based)** when:
  - You know specific sensitive values to remove
  - Rare diseases or protected categories exist
  - Regulatory requirements mandate specific exclusions
  
- **Use Strategy 2 (Risk-Based)** when:
  - K-anonymity risk scores are available
  - Automated privacy protection needed
  - Re-identification risk must be minimized
  
- **Use Strategy 3 (Custom)** when:
  - Complex business rules required
  - Compliance mandates specific combinations
  - GDPR right to be forgotten enforcement
  - HIPAA safe harbor requirements

**Next steps:**
- Test with your own datasets
- Adjust thresholds for optimal privacy/utility balance
- Combine with other anonymization techniques
- Deploy to production with audit trail monitoring

**Other Suppression Operations:**
- 📗 **Attribute Suppression**: Remove entire columns
- 📙 **Cell Suppression**: Replace individual cell values

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)